# CT Scan Pipeline

In [1]:
# Install required packages
!pip install pydicom opencv-python-headless torch torchvision

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.2/46.2 MB 25.2 MB/s  0:00:01 26.9 MB/s eta 0:00:01


In [2]:
import os
import cv2
import numpy as np
import pydicom
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

print('All imports successful')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

All imports successful
Using device: cpu


In [3]:
def get_series_paths(root_dir):
    series_paths = []
    for root, dirs, files in os.walk(root_dir):
        if any(f.lower().endswith('.dcm') for f in files):
            series_paths.append(root)
    return series_paths

root_path = 'Data/' # change to the actual data folder path
series_paths = get_series_paths(root_path)
print(f'Total CT series found: {len(series_paths)}')
print(f'Example path: {series_paths[0]}')

Total CT series found: 20
Example path: Data/LIDC-IDRI-0005/01-01-2000-NA-NA-06608/3001613.000000-NA-05796


In [46]:
# Load DICOM slices → 3D volume (num_slices, H, W)

def load_ct_series(series_path):
    slices = []
    for file in os.listdir(series_path):
        if file.lower().endswith(".dcm"):
            try:
                dicom = pydicom.dcmread(
                    os.path.join(series_path, file),
                    force=True    # to handle files missing DICOM header
                )
                
                if hasattr(dicom, 'pixel_array'):
                    slices.append(dicom)
            except Exception:
                pass

    if len(slices) == 0:
        raise ValueError(f"No valid DICOM slices found in {series_path}")

    slices.sort(key=lambda x: int(x.InstanceNumber))

    images = []
    for s in slices:
        image = s.pixel_array.astype(np.float32)
        if hasattr(s, "RescaleSlope") and hasattr(s, "RescaleIntercept"):
            image = image * s.RescaleSlope + s.RescaleIntercept
        if image.shape != (512, 512):
            image = cv2.resize(image, (512, 512))
        images.append(image)

    volume = np.stack(images, axis=0)
    return volume

volume = load_ct_series(series_paths[0])
print(f'Volume shape: {volume.shape}')
print(f'HU value range: {volume.min():.1f} to {volume.max():.1f}')

Volume shape: (2, 512, 512)
HU value range: 34.2 to 16344.0


In [5]:
# Normalize, select 3 middle slices, resize to 224x224

def normalize_volume(volume):
    return (volume - volume.min()) / (volume.max() - volume.min() + 1e-8)

def get_three_slices(volume):
    mid = volume.shape[0] // 2
    if volume.shape[0] < 3:
        return np.stack([volume[0]] * 3, axis=0)
    return volume[mid - 1 : mid + 2]

def resize_slices(slices, size=224):
    resized = [cv2.resize(slices[i], (size, size)) for i in range(3)]
    return np.stack(resized, axis=0)

volume = load_ct_series(series_paths[0])
volume = normalize_volume(volume)
slices = get_three_slices(volume)
slices = resize_slices(slices, size=224)
print(f'After preprocessing — shape: {slices.shape}')
print(f'Value range: {slices.min():.3f} to {slices.max():.3f}')

After preprocessing — shape: (3, 224, 224)
Value range: 0.021 to 0.999


In [6]:
# Dataset class that wraps everything above

class LIDCDataset(Dataset):
    def __init__(self, root_dir):
        self.series_paths = get_series_paths(root_dir)
        print(f'Dataset: {len(self.series_paths)} CT series')

    def __len__(self):
        return len(self.series_paths)

    def __getitem__(self, idx):
        volume = load_ct_series(self.series_paths[idx])
        volume = normalize_volume(volume)
        slices = get_three_slices(volume)
        slices = resize_slices(slices, size=224)
        return torch.tensor(slices, dtype=torch.float32)

dataset = LIDCDataset(root_path)
dataloader = DataLoader(dataset, batch_size=4, shuffle=True)

batch = next(iter(dataloader))
print(f'Batch shape: {batch.shape}')

Dataset: 20 CT series
Batch shape: torch.Size([4, 3, 224, 224])


## Tokenization

In [11]:
class PatchEmbed(nn.Module):
    def __init__(self, in_channels=3, embed_dim=1024, patch_size=16):
        super().__init__()
        self.proj = nn.Conv2d(in_channels, embed_dim,
                              kernel_size=patch_size, stride=patch_size)

    def forward(self, x):
        x = self.proj(x)          # (B, 1024, 14, 14)
        x = x.flatten(2)          # (B, 1024, 196)
        x = x.transpose(1, 2)     # (B, 196, 1024)
        return x

patch_embed = PatchEmbed().to(device)
tokens = patch_embed(batch)
print(f"tokens shape: {tokens.shape}")

tokens shape: torch.Size([4, 196, 1024])


## Feed Tokens into CTViT

In [13]:
from models.ctvit import CTViT

model = CTViT(
    embed_dim=1024,
    depth=24,
    num_heads=16,
).to(device)

print(f"tokens shape going into CTViT: {tokens.shape}")

output = model(
    tokens,
    window_size=(7, 7),
    window_block_indexes=[],
    spatial_size=(14, 14),
    cls_embed=False,
)
print(f'CTViT output shape: {output.shape}')

tokens shape going into CTViT: torch.Size([4, 196, 1024])
CTViT output shape: torch.Size([4, 196, 1024])


## Pretraining with 10 samples

In [16]:
# Pretraining
import torch.optim as optim

pretrain_subset = torch.utils.data.Subset(dataset, range(10))
pretrain_loader = DataLoader(pretrain_subset, batch_size=2, shuffle=True)

def mask_tokens(tokens, mask_ratio=0.75):
    B, N, C = tokens.shape  # (B, 196, 1024)
    num_masked = int(N * mask_ratio)

    # Random indices to mask
    noise = torch.rand(B, N, device=tokens.device)
    ids_shuffle = torch.argsort(noise, dim=1)
    mask_ids = ids_shuffle[:, :num_masked]

    # Build boolean mask
    mask = torch.zeros(B, N, dtype=torch.bool, device=tokens.device)
    mask.scatter_(1, mask_ids, True)

    # Zero out masked tokens
    masked_tokens = tokens.clone()
    masked_tokens[mask] = 0.0

    return masked_tokens, mask

patch_pixels = 16 * 16 * 3

reconstruction_head = nn.Linear(1024, patch_pixels).to(device)

optimizer = optim.Adam(
    list(model.parameters()) + list(patch_embed.parameters()) + list(reconstruction_head.parameters()),
    lr=1e-4
)

# Pretraining loop
model.train()
reconstruction_head.train()

num_epochs = 5

for epoch in range(num_epochs):
    total_loss = 0.0

    for batch in pretrain_loader:
        batch = batch.to(device)

        # Get tokens
        tokens = patch_embed(batch)

        # Mask tokens
        masked_tokens, mask = mask_tokens(tokens)

        # Forward through CTViT
        output = model(
            masked_tokens,
            window_size=(7, 7),
            window_block_indexes=[],
            spatial_size=(14, 14),
            cls_embed=False,
        )

        pred = reconstruction_head(output)

        target = reconstruction_head(tokens.detach())

        loss = ((pred - target) ** 2)[mask].mean()

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}/{num_epochs}  Loss: {total_loss/len(pretrain_loader):.4f}")

print("\nPretraining done! Saving weights...")
torch.save({
    'model': model.state_dict(),
    'patch_embed': patch_embed.state_dict(),
}, 'pretrained_weights.pth')
print("Saved → pretrained_weights.pth")

Epoch 1/5  Loss: 0.1837
Epoch 2/5  Loss: 0.0224
Epoch 3/5  Loss: 0.0169
Epoch 4/5  Loss: 0.0114
Epoch 5/5  Loss: 0.0083

Pretraining done! Saving weights...
Saved → pretrained_weights.pth


In [30]:
import pandas as pd

df = pd.read_excel("tcia-diagnosis-data-2012-04-20.xls")

# Clean column names
df.columns = [col.strip().replace('\n', ' ') for col in df.columns]

df = df.iloc[:, [0, 1]].copy()

df.columns = ["patient_id", "diagnosis"]

df = df.dropna(subset=["diagnosis"])
df["diagnosis"] = df["diagnosis"].astype(int)

print(df.head(10))
print(f"\nTotal labeled patients: {len(df)}")
print(f"\nClass distribution:\n{df['diagnosis'].value_counts().sort_index()}")

       patient_id  diagnosis
0  LIDC-IDRI-0068          3
1  LIDC-IDRI-0071          3
2  LIDC-IDRI-0072          2
3  LIDC-IDRI-0088          3
4  LIDC-IDRI-0090          2
5  LIDC-IDRI-0091          3
6  LIDC-IDRI-0100          3
7  LIDC-IDRI-0118          3
8  LIDC-IDRI-0124          3
9  LIDC-IDRI-0129          3

Total labeled patients: 157

Class distribution:
diagnosis
0    27
1    36
2    43
3    51
Name: count, dtype: int64


In [37]:
# Manually create labels for 10 patients

manual_labels = {
    "LIDC-IDRI-0001": 1,
    "LIDC-IDRI-0002": 1,
    "LIDC-IDRI-0003": 2,
    "LIDC-IDRI-0004": 1,
    "LIDC-IDRI-0005": 1,
    "LIDC-IDRI-0006": 2,
    "LIDC-IDRI-0007": 1,
    "LIDC-IDRI-0008": 1,
    "LIDC-IDRI-0009": 2,
    "LIDC-IDRI-0010": 1,
}

df_manual = pd.DataFrame(
    list(manual_labels.items()),
    columns=["patient_id", "diagnosis"]
)

print(df_manual)
print(f"\nClass distribution:\n{df_manual['diagnosis'].value_counts().sort_index()}")

       patient_id  diagnosis
0  LIDC-IDRI-0001          1
1  LIDC-IDRI-0002          1
2  LIDC-IDRI-0003          2
3  LIDC-IDRI-0004          1
4  LIDC-IDRI-0005          1
5  LIDC-IDRI-0006          2
6  LIDC-IDRI-0007          1
7  LIDC-IDRI-0008          1
8  LIDC-IDRI-0009          2
9  LIDC-IDRI-0010          1

Class distribution:
diagnosis
1    7
2    3
Name: count, dtype: int64


In [44]:
class LIDCLabeledDataset(Dataset):
    def __init__(self, root_dir, df):
        self.samples = []

        for _, row in df.iterrows():
            patient_id = str(row["patient_id"]).strip()
            label = int(row["diagnosis"])

            patient_dir = os.path.join(root_dir, patient_id)
            if not os.path.exists(patient_dir):
                continue

            series = get_series_paths(patient_dir)
            if len(series) == 0:
                continue

            self.samples.append((series[0], label))

        print(f"Labeled samples found: {len(self.samples)}")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        series_path, label = self.samples[idx]
        volume = load_ct_series(series_path)
        volume = normalize_volume(volume)
        slices = get_three_slices(volume)
        slices = resize_slices(slices, size=224)
        image = torch.tensor(slices, dtype=torch.float32)
        return image, torch.tensor(label, dtype=torch.long)

# whole dataset
labeled_dataset = LIDCLabeledDataset("/Volumes/Expansion/Data/manifest-1600709154662/LIDC-IDRI/", df)

train_size = int(0.8 * len(labeled_dataset))
val_size   = len(labeled_dataset) - train_size

train_dataset, val_dataset = torch.utils.data.random_split(
    labeled_dataset, [train_size, val_size]
)

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=4, shuffle=False)

print(f"Train: {len(train_dataset)}  Val: {len(val_dataset)}")

Labeled samples found: 157
Train: 125  Val: 32


In [47]:
# Fine-tuning

checkpoint = torch.load("pretrained_weights.pth", map_location=device)
model.load_state_dict(checkpoint["model"])
patch_embed.load_state_dict(checkpoint["patch_embed"])
print("Pretrained weights loaded")

num_classes = 4
classifier = nn.Linear(1024, num_classes).to(device)

class_counts = torch.tensor([27, 36, 43, 51], dtype=torch.float32)
class_weights = 1.0 / class_counts
class_weights = class_weights / class_weights.sum()
class_weights = class_weights.to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights)

# Optimizer
optimizer = torch.optim.Adam([
    {"params": model.parameters(),       "lr": 1e-5},
    {"params": patch_embed.parameters(), "lr": 1e-5},
    {"params": classifier.parameters(),  "lr": 1e-4},
])

# Fine-tuning loop
num_epochs = 10
model.train()
classifier.train()

for epoch in range(num_epochs):
    total_loss = 0.0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        tokens = patch_embed(images)

        output = model(
            tokens,
            window_size=(7, 7),
            window_block_indexes=[],
            spatial_size=(14, 14),
            cls_embed=False,
        )

        pooled = output.mean(dim=1)

        logits = classifier(pooled)
        loss   = criterion(logits, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}/{num_epochs}  Loss: {total_loss/len(train_loader):.4f}")

print("\nFine-tuning done!")

Pretrained weights loaded ✅
Epoch 1/10  Loss: 1.5069
Epoch 2/10  Loss: 1.4289
Epoch 3/10  Loss: 1.3980
Epoch 4/10  Loss: 1.4015
Epoch 5/10  Loss: 1.4205
Epoch 6/10  Loss: 1.4322
Epoch 7/10  Loss: 1.3598
Epoch 8/10  Loss: 1.4029
Epoch 9/10  Loss: 1.3271
Epoch 10/10  Loss: 1.4364

Fine-tuning done!


In [48]:
# Evaluation (F1 Score)
from sklearn.metrics import f1_score, classification_report

model.eval()
classifier.eval()

all_preds  = []
all_labels = []

with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device)

        tokens = patch_embed(images)
        output = model(
            tokens,
            window_size=(7, 7),
            window_block_indexes=[],
            spatial_size=(14, 14),
            cls_embed=False,
        )
        pooled = output.mean(dim=1)
        logits = classifier(pooled)

        preds = torch.argmax(logits, dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())

print(f"True labels : {all_labels}")
print(f"Predictions : {all_preds}")
print()

f1 = f1_score(all_labels, all_preds, average="macro", zero_division=0)
print(f"Macro F1 Score: {f1:.4f}")
print()

print(classification_report(
    all_labels, all_preds,
    target_names=["Unknown", "Benign", "Malignant-Primary", "Malignant-Metastatic"],
    zero_division=0
))

True labels : [np.int64(0), np.int64(0), np.int64(2), np.int64(3), np.int64(2), np.int64(0), np.int64(3), np.int64(3), np.int64(1), np.int64(1), np.int64(2), np.int64(3), np.int64(3), np.int64(3), np.int64(0), np.int64(0), np.int64(2), np.int64(1), np.int64(3), np.int64(3), np.int64(3), np.int64(2), np.int64(0), np.int64(3), np.int64(0), np.int64(3), np.int64(0), np.int64(3), np.int64(2), np.int64(1), np.int64(3), np.int64(0)]
Predictions : [np.int64(3), np.int64(3), np.int64(3), np.int64(3), np.int64(3), np.int64(3), np.int64(3), np.int64(0), np.int64(3), np.int64(3), np.int64(3), np.int64(0), np.int64(3), np.int64(3), np.int64(0), np.int64(0), np.int64(3), np.int64(0), np.int64(3), np.int64(3), np.int64(3), np.int64(3), np.int64(3), np.int64(3), np.int64(0), np.int64(3), np.int64(3), np.int64(0), np.int64(2), np.int64(0), np.int64(3), np.int64(2)]

Macro F1 Score: 0.2936

                      precision    recall  f1-score   support

             Unknown       0.38      0.33      0.3